# 02 · Calibración de N, U y umbral de fraude (§8.3)

Valores vigentes en `services/src/sofia_services/settings.py` (SIM): `DISPUTE_WINDOW_DAYS=90`,
`AMOUNT_THRESHOLD_USD=500`, `FRAUD_SCORE_THRESHOLD=0.8`. Datos: silver de una corrida local del pipeline
(`python -m sofia_data.pipeline --source local`), con gold idéntico a la corrida de OPS.

Salida: `analysis/results/policy_calibration.json`. Los valores finales los decide el equipo (`value`); aquí va la evidencia.

In [ ]:
from common import connect, save_result, show

con = connect()

## 1. N: ¿se puede ligar una disputa con su transacción?

`complaints` no trae `transaction_id` (§14). Se prueba (a) el producto afectado y (b) el monto reclamado contra las
transacciones del propio cliente.

In [ ]:
con.sql("""CREATE OR REPLACE TEMP TABLE disputas AS
    SELECT complaint_id, customer_id, affected_product_id AS pid, creation_date::TIMESTAMP AS cdate,
           claimed_amount, currency
    FROM complaints WHERE subcategory IN ('Cargo no reconocido', 'Cobro indebido')""")
link = show(con, "¿El producto afectado es del cliente que reclama?", """
    SELECT count(*) AS disputas, count(d.pid) AS con_producto, count(p.product_id) AS producto_existe,
           count(*) FILTER (WHERE p.customer_id = d.customer_id) AS producto_del_cliente
    FROM disputas d LEFT JOIN products p ON p.product_id = d.pid""")
amount = show(con, "Match de monto contra transacciones del propio cliente (365 días previos, ±1%)", """
    WITH l AS (
        SELECT d.complaint_id, d.claimed_amount, d.currency AS ccur, t.amount, t.currency AS tcur
        FROM disputas d JOIN transactions t ON t.customer_id = d.customer_id
         AND t.transaction_date::TIMESTAMP BETWEEN d.cdate - INTERVAL 365 DAY AND d.cdate)
    SELECT count(DISTINCT complaint_id) FILTER (WHERE claimed_amount IS NOT NULL) AS con_monto,
           count(DISTINCT complaint_id) FILTER (WHERE claimed_amount IS NOT NULL AND tcur = ccur
               AND abs(amount - claimed_amount) <= 0.01 * claimed_amount) AS con_match
    FROM l""")

**Resultado:** el producto afectado existe pero nunca es del cliente que reclama, y el match de monto está al nivel
del azar. El dataset sintético no liga disputas con transacciones, así que N **no se puede calibrar con datos**:
queda como supuesto de política del equipo (CON-02) y se reporta como limitación. Como referencia indirecta, días
desde la última transacción del cliente hasta la queja:

In [ ]:
lag = show(con, "Días desde la última transacción del cliente hasta la queja (proxy, no calibra N)", """
    WITH l AS (
        SELECT d.complaint_id, min(date_diff('day', t.transaction_date::TIMESTAMP, d.cdate)) AS lag
        FROM disputas d JOIN transactions t ON t.customer_id = d.customer_id
         AND t.transaction_date::TIMESTAMP BETWEEN d.cdate - INTERVAL 365 DAY AND d.cdate
        GROUP BY 1)
    SELECT count(*) AS n, quantile_cont(lag, [0.5, 0.75, 0.9, 0.95]) AS p50_75_90_95 FROM l""")

## 2. U: montos en USD

`amount_usd` viene vacío en todas las transacciones en USD (ya están en USD): el monto en USD es
`coalesce(amount_usd, amount si currency = 'USD')`. Esto explica casi todo el aviso de "57% nulos".

In [ ]:
usd = "coalesce(amount_usd, CASE WHEN currency = 'USD' THEN amount END)::DOUBLE"
purchases = show(con, "Compras aprobadas en USD", f"""
    SELECT count(*) AS n, quantile_cont({usd}, [0.5, 0.75, 0.8, 0.9, 0.95]) AS p50_75_80_90_95,
           round(avg(({usd} > 250)::INT), 3) AS share_sobre_250, round(avg(({usd} > 500)::INT), 3) AS share_sobre_500
    FROM transactions
    WHERE transaction_status = 'Approved' AND transaction_type = 'Purchase' AND {usd} IS NOT NULL""")  # noqa: S608

In [ ]:
claimed = show(con, "Monto reclamado en disputas, en USD (tipo de cambio del día)", """
    WITH c AS (
        SELECT c.claimed_amount * CASE WHEN c.currency = 'USD' THEN 1 ELSE f.exchange_rate END AS usd
        FROM complaints c
        LEFT JOIN daily_exchange_rates f
          ON f.source_currency = c.currency AND f.target_currency = 'USD' AND f.date = c.creation_date::DATE
        WHERE c.subcategory IN ('Cargo no reconocido', 'Cobro indebido') AND c.claimed_amount IS NOT NULL)
    SELECT count(usd) AS n, quantile_cont(usd, [0.5, 0.75, 0.8, 0.9, 0.95]) AS p50_75_80_90_95,
           round(avg((usd > 500)::INT), 3) AS share_sobre_500
    FROM c""")

**Resultado:** las compras sintéticas llegan hasta ~500 USD, así que con U=500 ninguna compra va a humano por
monto; sobre montos reclamados, U=500 manda ~23% a humano. U=250 parte las compras por la mitad.

## 3. Umbral de fraude

`fraud_score` va de **0 a 100**, no de 0 a 1. El umbral vigente (0.8) marcaría ~97% de las transacciones como
sospechosas apenas bank-api lea gold.

In [ ]:
fraud = show(con, "Precisión / recall por umbral (escala 0–100)", """
    SELECT th, round(sum((fraud_score >= th AND is_fraud)::INT) / nullif(sum((fraud_score >= th)::INT), 0), 3) AS prec,
           round(sum((fraud_score >= th AND is_fraud)::INT) / sum(is_fraud::INT), 3) AS rec,
           round(100 * avg((fraud_score >= th)::INT), 2) AS pct_marcada
    FROM transactions, (VALUES (0.8), (30), (40), (50), (70), (80)) v(th)
    WHERE fraud_score IS NOT NULL AND is_fraud IS NOT NULL GROUP BY th ORDER BY th""")

### Umbral por costo esperado

No hay norma que fije el umbral: la práctica es minimizar `costo = c_FN · fraudes no detectados + c_FP · revisiones de más`,
con un fraude perdido ≈ 40–50× una revisión de más (literatura de detección de fraude sensible a costo).

In [ ]:
cost = show(con, "Costo esperado por umbral (escala 0–100)", """
    WITH g AS (
        SELECT th, sum((fraud_score >= th AND is_fraud)::INT) AS tp,
               sum((fraud_score >= th AND NOT is_fraud)::INT) AS fp,
               sum((fraud_score < th AND is_fraud)::INT) AS fn
        FROM transactions, (SELECT unnest(range(20, 46, 2)) AS th)
        WHERE fraud_score IS NOT NULL AND is_fraud IS NOT NULL GROUP BY th)
    SELECT th, tp, fp, fn, round(tp / (tp + fp), 3) AS prec, round(tp / (tp + fn), 3) AS rec,
           50 * fn + fp AS costo_50_1, 40 * fn + fp AS costo_40_1
    FROM g ORDER BY th""")

**Resultado:** el mínimo de costo está en **30** con ambas razones (recall 0.69, precisión 0.80, 0.08% a revisión).
Con 40 se pierden ~360 fraudes más para ahorrar 609 revisiones. Ojo: las transacciones legítimas del dataset sintético
tienen score ≤ ~30, por eso el salto abrupto entre 28 y 30; un banco real recalibraría con sus datos.
`is_fraud` es la etiqueta del dataset: un banco real no la tiene al momento de la disputa (limitación).

## Decisión del equipo (2026-10-04)

| Parámetro | Valor | Justificación |
|---|---|---|
| **N** | 90 días | No calibrable con datos. México: 90 días naturales para objetar cargos no reconocidos (Condusef, LTOSF art. 23); dentro de los 120 días de chargeback de Visa/Mastercard. Argentina es más estricta (30 días desde el resumen, Ley 25.065 art. 26): ventana por país como paso a producción |
| **U** | 500 USD | Ninguna norma fija un monto. Automatiza casi todas las compras (p95 475 USD) y manda a humano el 23% de los montos reclamados |
| **Umbral de fraude** | 30 en gold = 0.30 en la política | Mínimo costo esperado (arriba). bank-api normaliza gold de 0–100 a 0–1 al cargar |

In [ ]:
save_result("policy_calibration", {
    "dispute_transaction_link": {"product": link.to_dicts()[0], "amount": amount.to_dicts()[0]},
    "N_days": {"value": 90, "current": 90,
               "basis": "MX 90 días naturales (Condusef/LTOSF art. 23); Visa/MC 120 días",
               "proxy_lag_days": lag.to_dicts()[0],
               "rationale": "No calibrable: el dataset no liga disputas con transacciones. Supuesto de política."},
    "U_usd": {"value": 500, "current": 500, "approved_purchases": purchases.to_dicts()[0],
              "claimed_amounts": claimed.to_dicts()[0]},
    "fraud_score_threshold": {"value": 30, "policy_value_unit_scale": 0.30, "previous": 0.8, "scale": "0-100",
                              "tradeoff": fraud.to_dicts(), "cost_curve": cost.to_dicts(),
                              "rationale": "mínimo costo esperado con c_FN/c_FP = 40–50"},
    "data_quality_notes": {"amount_usd": "vacío en filas USD; usar coalesce(amount_usd, amount si USD)"},
})